In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("last_mile_delivery_dataset.csv")
print("Dataset Shape:", df.shape)
df = df.drop_duplicates()
df = df.dropna()
if 'order_date' in df.columns:
    df['order_date'] = pd.to_datetime(df['order_date'])
if 'order_time' in df.columns:
    df['order_time'] = pd.to_datetime(df['order_time'])
if 'order_time' in df.columns:
    df['hour'] = df['order_time'].dt.hour
print("\n========== Q1 ==========")
traffic_period = []
for hour in df['hour']:
    if (hour >= 8 and hour <= 10) or (hour >= 17 and hour <= 20):
        traffic_period.append("Peak")
    else:
        traffic_period.append("Off-Peak")
df['traffic_period'] = traffic_period
peak = df[df['traffic_period'] == 'Peak']
offpeak = df[df['traffic_period'] == 'Off-Peak']
peakmean = peak['delay_mins'].mean()
offmean = offpeak['delay_mins'].mean()
difference = peakmean - offmean
print("Peak Average Delay:", round(peakmean),)
print("Off-Peak Average Delay:", round(offmean),)
print("Difference:", round(difference), "minutes")

print("\n========== Q2 ==========")
weather_delay = df.groupby('weather_condition')['delay_mins'].median()
print("Median Delay by Weather")
print(weather_delay)
rainorder = df[df['weather_condition'] == 'Rain']
raindelay = rainorder.groupby('order_type')['delay_mins'].median()
raindelay = raindelay.sort_values(ascending=False)
print("\nRain Impact by Order Type")
print(raindelay)
affected = raindelay.idxmax()
print("\nMost Affected Order Type:")
print(affected)
weather_delay.plot(kind='bar')
plt.title("Median Delay by Weather")
plt.xlabel("Weather Condition")
plt.ylabel("Median Delay (mins)")
plt.show()

print("\n========== Q3 ==========")
low = df[df['rider_experience_yrs'] < 2]
high = df[df['rider_experience_yrs'] > 4]
low_mean = low['delay_mins'].mean()
high_mean = high['delay_mins'].mean()
print("Average Delay (<2 years):", round(low_mean,2))
print("Average Delay (>4 years):", round(high_mean,2))
difference = low_mean - high_mean
print("Difference:", round(difference,2), "minutes")
if difference > 0:
    print("Less experienced riders have higher delays.")
else:
    print("Experience has little impact.")

print("\n========== Q4 ==========")
ontime = []
for delay in df['delay_mins']:
    if delay <= 10:
        ontime.append(1)
    else:
        ontime.append(0)
df['on_time'] = ontime
city_ontime = df.groupby('city')['on_time'].mean() * 100
print("On-Time Rate by City")
print(city_ontime)
df['month'] = df['order_date'].dt.month
monthly_delay = df.groupby('month')['delay_mins'].mean()
print("\nMonthly Average Delay")
print(monthly_delay)
vehicle_delay = df.groupby('vehicle_type')['delay_mins'].mean()
print("\nVehicle Type Delay")
print(vehicle_delay)
fig, axes = plt.subplots(1,3,figsize=(18,5))
city_ontime.sort_values().plot(kind='barh', ax=axes[0])
axes[0].set_title("City-wise On-Time Rate")
monthly_delay.plot(marker='o', ax=axes[1])
axes[1].set_title("Monthly Delay Trend")
axes[1].set_ylabel("Delay (mins)")
vehicle_delay.plot( kind='bar', ax=axes[2])
axes[2].set_title("Vehicle Type Comparison")
axes[2].set_ylabel("Average Delay")
plt.tight_layout()
plt.show()
worst_city = city_ontime.idxmin()
worst_vehicle = vehicle_delay.idxmax()
print("\nWorst City:", worst_city)
print("Worst Vehicle:", worst_vehicle)
print( "\nBiggest Operational Fix:")
print("Increase rider availability during peak hours "
    "(8-10 AM and 5-8 PM) and improve routing during "
    "Rain/Fog conditions, especially in the worst-performing city.")
